# **Warung Madura Sales Forecasting**

In [54]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from _support_function import create_window, splitting_data, result_reporting 

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import pickle
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### **Data Loading**

In [55]:
data_for_forecasting = pd.read_csv('data/data_for_forecasting.csv', index_col = 'order_date')

In [56]:
data_for_forecasting

,P001,P002,P003,P004,P005,P006,P007,P008,P009,P010,P011,P012,P013,P014,P015,P016,P017,P018,P019,P020
order_date,,,,,,,,,,,,,,,,,,,,
2025-01-01,3,0,1,0,5,1,0,0,0,0,0,0,2,0,0,1,0,4,0,0
2025-01-02,6,4,4,0,8,5,1,8,3,3,0,5,4,4,5,2,5,0,2,5
2025-01-03,0,0,3,3,0,3,2,0,6,1,1,6,1,3,1,0,1,2,3,0
2025-01-04,0,3,0,2,3,2,0,2,2,6,5,2,0,1,0,0,5,0,0,3
2025-01-05,6,1,6,1,0,4,1,1,1,1,1,3,0,0,0,0,4,3,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27,0,5,6,1,0,0,2,2,8,0,0,3,6,4,2,3,3,3,6,0
2025-12-28,5,0,0,2,0,12,5,1,0,2,2,5,2,2,0,1,4,6,3,2
2025-12-29,0,0,3,2,0,0,2,0,0,1,2,3,3,7,0,6,1,3,2,1


### **Creating Window**

In [57]:
data_for_transformation_array = np.array(data_for_forecasting)

**Pembuatan Windows Data sebagai Fitur Prediksi**

In [58]:
def create_window(data, window_size):
  feature_data = []
  pred_data = []

  for i in range(window_size, len(data)) :
    seq_data  = data[i - window_size : i]
    prediction_data = data[i]

    feature_data.append(seq_data)
    pred_data.append(prediction_data)

  return feature_data, pred_data

In [59]:
X, y = create_window(data_for_transformation_array, 7)

### **Splitting_data**

**Membagi Data menjadi 3 Bagian - Train, Valid, dan Test**

In [60]:
# Splitting Data
train_data, valid_data, test_data = splitting_data(X, y, 0.7, 0.2, 0.1)

X_train, y_train = train_data
X_valid, y_valid = valid_data
X_test, y_test   = test_data

In [61]:
# Convert to Array
X_train = np.array(X_train)
y_train = np.array(y_train)
X_valid = np.array(X_valid)
y_valid = np.array(y_valid)
X_test  = np.array(X_test)
y_test  = np.array(y_test)

In [62]:
X_train.shape, y_train.shape, X_valid.shape, y_valid.shape, X_test.shape, y_test.shape

((250, 7, 20), (250, 20), (71, 7, 20), (71, 20), (37, 7, 20), (37, 20))

In [63]:
X_train[:,:,0].shape

(250, 7)

## **Machine Learning Development - LSTM**

### **Data Preparation**

#### **Feature Scaling**

In [64]:
X_train_scaled = X_train.copy()
X_valid_scaled = X_valid.copy()
X_test_scaled  = X_test.copy()

**Mengscaling data tiap produk**

In [65]:
scaler_for_product = {}

# Mapping
for i in range(X_train.shape[2]) :
  scaler_for_product[i] = StandardScaler()
  prod_data = np.array(X_train_scaled)[:,:,i].reshape(-1, 1)
  scaler_for_product[i].fit(prod_data)

In [66]:
# Transform
for i in range(len(scaler_for_product.keys())) :
  train_shape = X_train[:,:,i].shape
  valid_shape = X_valid[:,:,i].shape
  test_shape  = X_test[:,:,i].shape

  X_train_scaled[:,:,i] = scaler_for_product[i].transform(X_train_scaled[:,:,i].reshape(-1, 1)).reshape(train_shape)
  X_valid_scaled[:,:,i] = scaler_for_product[i].transform(X_valid_scaled[:,:,i].reshape(-1, 1)).reshape(valid_shape)
  X_test_scaled[:,:,i]  = scaler_for_product[i].transform(X_test_scaled[:,:,i].reshape(-1, 1)).reshape(test_shape)

#### **Data Optimization**

**Merubah data menjadi tensor**

In [67]:
train_data_tensor  = torch.tensor(X_train_scaled, dtype = torch.float32)
train_label_tensor = torch.tensor(y_train, dtype = torch.float32)

valid_data_tensor  = torch.tensor(X_valid_scaled, dtype = torch.float32)
valid_label_tensor = torch.tensor(y_valid, dtype = torch.float32)

test_data_tensor   = torch.tensor(X_test_scaled, dtype = torch.float32)
test_label_tensor  = torch.tensor(y_test, dtype = torch.float32)


**Merubah tensor menjadi tensor dataset**

In [68]:
train_dataset = TensorDataset(train_data_tensor, train_label_tensor)
valid_dataset = TensorDataset(valid_data_tensor, valid_label_tensor)
test_dataset  = TensorDataset(test_data_tensor, test_label_tensor)

**Merubah tensor dataset menjadi data loader untuk model**

In [69]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
valid_loader = DataLoader(valid_dataset, batch_size = 32, shuffle = False)
test_loader  = DataLoader(test_dataset, batch_size = 32, shuffle = False)

### **Modelling**

#### **Setup Model**

In [70]:
X_train_scaled.shape, y_train.shape, X_valid_scaled.shape, y_valid.shape, X_test_scaled.shape, y_test.shape

((250, 7, 20), (250, 20), (71, 7, 20), (71, 20), (37, 7, 20), (37, 20))

In [71]:
input_size  = 20
output_size = 20

In [72]:
class LSTM_Model(nn.Module) :
  def __init__(self, input_size = 20, hidden_size = 64, output_size = 20, layer_size = 2, dropout = 0.2) :
      super().__init__()

      self.input_size = input_size
      self.hidden_size = hidden_size
      self.output_size = output_size
      self.layer_size = layer_size

      self.lstm = nn.LSTM(
                        input_size  = input_size,
                        hidden_size = hidden_size,
                        num_layers  = layer_size,
                        dropout     = dropout if layer_size > 1 else 0,
                        batch_first = True
      )

      self.fc = nn.Linear(
                          in_features  = hidden_size,
                          out_features = output_size
      )

  def forward(self, x):
         output, (hidden, cell) = self.lstm(x)
         last_output = output[:, -1, :]
         prediction = self.fc(last_output)
         return prediction

#### **Training Model**

**Setup Fungsi untuk Training**

In [73]:
class TrainingModel:
    def __init__(
        self,
        model,
        optimizer,
        criterion,
        scheduler=None
    ):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.scheduler = scheduler

    def training_loop(self):

        self.model.train()

        total_loss = 0

        for X_batch, y_batch in self.train_loader:

            self.optimizer.zero_grad()
            
            pred = self.model(X_batch)
            loss = self.criterion(pred, y_batch)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()

        return total_loss / len(self.train_loader)

    def validation_loop(self):

        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for X_batch, y_batch in self.valid_loader:
                pred = self.model(X_batch)
                loss = self.criterion(pred, y_batch)
                total_loss += loss.item()

        return total_loss / len(self.valid_loader)

    def fit(
        self,
        train_loader,
        valid_loader,
        num_epochs=100,
        patience=20,
        min_delta=1e-5
    ):

        self.train_loader = train_loader
        self.valid_loader = valid_loader

        train_losses = []
        valid_losses = []

        best_val_loss = float("inf")
        patience_counter = 0

        for epoch in range(num_epochs):

            train_loss = self.training_loop()
            valid_loss = self.validation_loop()

            train_losses.append(train_loss)
            valid_losses.append(valid_loss)

            # Scheduler callback
            if self.scheduler is not None:
                self.scheduler.step(valid_loss)

            # ModelCheckpoint callback
            if valid_loss < (best_val_loss - min_delta):

                best_val_loss = valid_loss

                torch.save(
                    self.model.state_dict(),
                    "result/model/lstm_best_model.pt"
                )

                patience_counter = 0

            else:

                patience_counter += 1

            print(
                f"Epoch [{epoch+1}/{num_epochs}] | "
                f"Train Loss: {train_loss:.6f} | "
                f"Valid Loss: {valid_loss:.6f} | "
                f"Patience: {patience_counter}/{patience}"
            )

            # EarlyStopping callback
            if patience_counter >= patience:

                print(
                    f"Early stopping at epoch {epoch+1}"
                )

                break

        return train_losses, valid_losses

**Setup Training**

In [74]:
# Set Model Architecture
model_lstm = LSTM_Model(
              input_size  = 20,
              hidden_size = 64,
              layer_size  = 2,
              output_size = 20,
              dropout     = 0.2
)

# Set Loss Function
criterion = nn.MSELoss()

# Set Optimizer
optimizer = torch.optim.Adam(
                    model_lstm.parameters(),
                    lr = 0.001
)

In [75]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = "min",
    factor = 0.5,
    patience = 5,
    min_lr = 1e-4
)

**Training Model**

In [76]:
num_epochs = 200
patience = 20
factor = 0.5
scheduler_patience = 5
min_delta = 1e-4

In [77]:
training_LSTM = TrainingModel(
            model = model_lstm,
            optimizer = optimizer,
            criterion = criterion,
            scheduler = scheduler
)

training_LSTM.fit(
      train_loader = train_loader,
      valid_loader = valid_loader,
      num_epochs = num_epochs,
      patience = patience
)

Epoch [1/200] | Train Loss: 9.163930 | Valid Loss: 8.582735 | Patience: 0/20
Epoch [2/200] | Train Loss: 8.638778 | Valid Loss: 7.792892 | Patience: 0/20
Epoch [3/200] | Train Loss: 7.181761 | Valid Loss: 5.795747 | Patience: 0/20
Epoch [4/200] | Train Loss: 5.412839 | Valid Loss: 5.070173 | Patience: 0/20
Epoch [5/200] | Train Loss: 5.027010 | Valid Loss: 5.063070 | Patience: 0/20
Epoch [6/200] | Train Loss: 4.949269 | Valid Loss: 4.963561 | Patience: 0/20
Epoch [7/200] | Train Loss: 4.894521 | Valid Loss: 4.927509 | Patience: 0/20
Epoch [8/200] | Train Loss: 4.888097 | Valid Loss: 4.903655 | Patience: 0/20
Epoch [9/200] | Train Loss: 4.933163 | Valid Loss: 4.891679 | Patience: 0/20
Epoch [10/200] | Train Loss: 4.910649 | Valid Loss: 4.903674 | Patience: 1/20
Epoch [11/200] | Train Loss: 4.896386 | Valid Loss: 4.907357 | Patience: 2/20
Epoch [12/200] | Train Loss: 4.895506 | Valid Loss: 4.908559 | Patience: 3/20
Epoch [13/200] | Train Loss: 4.906990 | Valid Loss: 4.905389 | Patience: 

([9.163930416107178,
  8.638777613639832,
  7.18176144361496,
  5.412838876247406,
  5.027010023593903,
  4.949269413948059,
  4.89452064037323,
  4.8880966901779175,
  4.933162808418274,
  4.910649478435516,
  4.896385669708252,
  4.895506203174591,
  4.906989932060242,
  4.8791550397872925,
  4.889205634593964,
  4.891749382019043,
  4.886028170585632,
  4.884095251560211,
  4.879601955413818,
  4.927883863449097,
  4.894842088222504,
  4.871852278709412,
  4.906899690628052,
  4.8850215673446655,
  4.880460739135742,
  4.8763198256492615,
  4.891491889953613,
  4.891397953033447,
  4.9034974575042725],
 [8.582734743754068,
  7.79289182027181,
  5.79574712117513,
  5.070172945658366,
  5.063069820404053,
  4.963561216990153,
  4.92750898996989,
  4.9036545753479,
  4.891678651173909,
  4.903674443562825,
  4.90735658009847,
  4.9085586865743,
  4.905389149983724,
  4.903690338134766,
  4.907756010691325,
  4.908863703409831,
  4.9100314776102705,
  4.907650152842204,
  4.905131657918

#### **Evaluation Model**

**Setup Evaluation**

In [79]:
model_lstm = LSTM_Model(
              input_size  = 20,
              hidden_size = 64,
              layer_size  = 2,
              output_size = 20,
              dropout     = 0.2
)

state_dict = torch.load('result/model/lstm_best_model.pt',  weights_only=True)
model_lstm.load_state_dict(state_dict)

<All keys matched successfully>

In [80]:
def prediction(model, data_loader) :
    predictions = []
    actuals = []

    model.eval()

    with torch.no_grad():

        for X_batch, y_batch in data_loader:
            prediction = model(X_batch)

            predictions.append(
                prediction.cpu().numpy()
            )

            actuals.append(
                y_batch.numpy()
            )

    return predictions, actuals

**Melihat Hasil Prediksi dan Aktual dari Data Test**

In [81]:
y_train_lstm_pred, y_train_actuals = prediction(model_lstm, train_loader)
y_valid_lstm_pred, y_valid_actuals = prediction(model_lstm, valid_loader)
y_test_lstm_pred, y_test_actuals = prediction(model_lstm, test_loader)

In [82]:
y_train_lstm_pred = np.concatenate(y_train_lstm_pred)
y_valid_lstm_pred = np.concatenate(y_valid_lstm_pred)
y_test_lstm_pred  = np.concatenate(y_test_lstm_pred)

**Metrics**

##### **MAE**

In [83]:
lstm_mae_train = mean_absolute_error(y_train_lstm_pred, y_train)
lstm_mae_valid = mean_absolute_error(y_valid_lstm_pred, y_valid)
lstm_mae_test = mean_absolute_error(y_test_lstm_pred, y_test)

In [84]:
lstm_mae_train, lstm_mae_valid, lstm_mae_test

(1.7550026178359985, 1.799778938293457, 1.699449896812439)

##### **MSE**

In [85]:
lstm_mse_train = mean_squared_error(y_train_lstm_pred, y_train)
lstm_mse_valid = mean_squared_error(y_valid_lstm_pred, y_valid)
lstm_mse_test = mean_squared_error(y_test_lstm_pred, y_test)

In [86]:
lstm_mse_train, lstm_mse_valid, lstm_mse_test

(4.88535213470459, 5.09988260269165, 4.310361862182617)

##### **R2-Score**

In [87]:
lstm_r2s_train = r2_score(y_train_lstm_pred, y_train)
lstm_r2s_valid = r2_score(y_valid_lstm_pred, y_valid)
lstm_r2s_test  = r2_score(y_test_lstm_pred, y_test)

In [88]:
lstm_r2s_train, lstm_r2s_valid, lstm_r2s_test

(-78966.2421875, -54294.35546875, -49836.3984375)

#### **Future Prediction**

**Memprediksi Jumlah Penjualan tiap Produk pada 30 Hari ke Depan**

In [89]:
future_days = 30

**Mengambil 7 data terakhir sebagai input**

In [90]:
last_2_7_days = X_test_scaled[-1][1:]
last_1_day = [scaler_for_product[i].transform(y_test[-1][i].reshape(-1, 1))[0][0] for i in range(20)]
last_7_days = np.array([
                        last_2_7_days[i] if i < 6 else last_1_day for i in range(len(last_2_7_days) + 1)
])
last_7_days = torch.tensor(last_7_days, dtype = torch.float32).unsqueeze(0)

**Memprediksi sekaligus menjadikan data hasil prediksi sebagai inputan**

In [91]:
lstm_future_predictions = []

model_lstm.eval()

with torch.no_grad():

    for _ in range(future_days):

        prediction = model_lstm(last_7_days)
        lstm_future_predictions.append(prediction.squeeze(0).numpy())

        prediction_for_window = []

        for i in range(20):
            pred_original = prediction[0, i].item()
            pred_scaled = (scaler_for_product[i].transform([[pred_original]]).item())
            prediction_for_window.append(pred_scaled)

        prediction_for_window = torch.tensor(prediction_for_window,dtype = torch.float32)
        prediction_for_window = (prediction_for_window.unsqueeze(0).unsqueeze(1))

        last_7_days = torch.cat((last_7_days[:, 1:, :], prediction_for_window),dim = 1)

In [92]:
lstm_future_predictions = np.array(lstm_future_predictions)

**Membuat Data untuk Laporan Hasil Prediksi**

In [93]:
prod_unq = pd.Series(data_for_forecasting.columns)
prod_unq = prod_unq.sort_values().reset_index(drop = True).to_list()
prod_unq

['P001',
 'P002',
 'P003',
 'P004',
 'P005',
 'P006',
 'P007',
 'P008',
 'P009',
 'P010',
 'P011',
 'P012',
 'P013',
 'P014',
 'P015',
 'P016',
 'P017',
 'P018',
 'P019',
 'P020']

In [94]:
tanggal_prediksi = pd.date_range(
    start   = '2026-01-01',
    periods = future_days,
    freq    = 'D'
)

lstm_future_prediction_df = pd.DataFrame(
                  {'order_date' : tanggal_prediksi}
                  |
                  {
                    f'{prod_unq[i]}' : lstm_future_predictions[:, i].astype(int) for i in range(20)
                  }
)

lstm_future_prediction_data = lstm_future_prediction_df.melt(
                                        id_vars = 'order_date',
                                        var_name = 'product_id',
                                        value_name = 'quantity'
)
lstm_future_prediction_data['model'] = 'LSTM'
lstm_future_prediction_data['type'] = 'Prediction'
lstm_future_prediction_data['part']  = 'Future'

In [95]:
lstm_future_prediction_data

,order_date,product_id,quantity,model,type,part
0,2026-01-01,P001,1,LSTM,Prediction,Future
1,2026-01-02,P001,1,LSTM,Prediction,Future
2,2026-01-03,P001,1,LSTM,Prediction,Future
3,2026-01-04,P001,1,LSTM,Prediction,Future
4,2026-01-05,P001,1,LSTM,Prediction,Future
...,...,...,...,...,...,...
595,2026-01-26,P020,0,LSTM,Prediction,Future
596,2026-01-27,P020,0,LSTM,Prediction,Future
597,2026-01-28,P020,0,LSTM,Prediction,Future
598,2026-01-29,P020,0,LSTM,Prediction,Future


In [96]:
lstm_future_prediction_df

,order_date,P001,P002,P003,P004,P005,P006,P007,P008,P009,...,P011,P012,P013,P014,P015,P016,P017,P018,P019,P020
0,2026-01-01,1,2,2,2,1,1,2,1,2,...,2,1,1,2,2,1,1,2,2,1
1,2026-01-02,1,2,2,2,1,1,2,1,2,...,2,1,1,2,2,1,1,2,2,2
2,2026-01-03,1,2,2,2,1,1,2,1,2,...,2,1,1,2,2,1,1,2,2,2
3,2026-01-04,1,2,2,2,1,1,2,1,2,...,2,1,1,2,2,1,1,2,2,1
4,2026-01-05,1,1,1,2,1,1,2,1,1,...,2,1,1,2,2,1,1,2,1,1
5,2026-01-06,1,1,1,2,1,1,2,1,1,...,1,1,1,2,2,1,1,2,1,1
6,2026-01-07,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
7,2026-01-08,1,1,1,2,1,1,2,1,1,...,1,1,1,2,2,1,1,2,1,1
8,2026-01-09,1,1,1,2,1,1,2,1,1,...,1,1,1,2,2,1,1,2,1,1
9,2026-01-10,1,1,1,2,1,1,1,1,1,...,1,1,1,1,2,1,1,2,1,1


In [97]:
lstm_future_prediction_data['product_id'].unique()

<StringArray>
['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009',
 'P010', 'P011', 'P012', 'P013', 'P014', 'P015', 'P016', 'P017', 'P018',
 'P019', 'P020']
Length: 20, dtype: str

### **Reporting**

#### **Model Prediction**

In [98]:
train_size = len(X_train)
valid_size = len(X_valid)
test_size  = len(X_test)

In [99]:
idx = data_for_forecasting[7:].index

In [100]:
train_idx = idx[:train_size]
valid_idx = idx[train_size:train_size + valid_size]
test_idx  = idx[train_size + valid_size:]

**Actual Data**

In [101]:
actual_result_reporting = result_reporting(
    data_train = y_train,
    data_valid = y_valid,
    data_test = y_test,
    columns = data_for_forecasting.columns,
    data_type = 'Actual', idx = (train_idx, valid_idx, test_idx),
    melt_var = {'id_vars' : 'order_date', 'var_name' : 'product_id', 'value_name' : 'quantity'}
)
actual_result_reporting

,order_date,product_id,quantity,model,type,part
0,2025-01-08,P001,4,-,Actual,Train
1,2025-01-09,P001,0,-,Actual,Train
2,2025-01-10,P001,1,-,Actual,Train
3,2025-01-11,P001,0,-,Actual,Train
4,2025-01-12,P001,2,-,Actual,Train
...,...,...,...,...,...,...
735,2025-12-27,P020,0,-,Actual,Test
736,2025-12-28,P020,2,-,Actual,Test
737,2025-12-29,P020,1,-,Actual,Test
738,2025-12-30,P020,5,-,Actual,Test


**LSTM Prediction Data for Reporting**

In [102]:
lstm_result_reporting = result_reporting(
    data_train = y_train_lstm_pred,
    data_valid = y_valid_lstm_pred,
    data_test = y_test_lstm_pred,
    columns = data_for_forecasting.columns,
    model_name = 'LSTM',
    data_type = 'Prediction', idx = (train_idx, valid_idx, test_idx),
    melt_var = {'id_vars' : 'order_date', 'var_name' : 'product_id', 'value_name' : 'quantity'}
)
lstm_result_reporting

,order_date,product_id,quantity,model,type,part
0,2025-01-08,P001,1.949117,LSTM,Prediction,Train
1,2025-01-09,P001,1.962582,LSTM,Prediction,Train
2,2025-01-10,P001,1.969131,LSTM,Prediction,Train
3,2025-01-11,P001,1.962889,LSTM,Prediction,Train
4,2025-01-12,P001,1.949242,LSTM,Prediction,Train
...,...,...,...,...,...,...
735,2025-12-27,P020,2.011674,LSTM,Prediction,Test
736,2025-12-28,P020,2.021148,LSTM,Prediction,Test
737,2025-12-29,P020,2.030626,LSTM,Prediction,Test
738,2025-12-30,P020,2.021528,LSTM,Prediction,Test


In [103]:
lstm_prediction_result_reporting = pd.concat([lstm_result_reporting, lstm_future_prediction_data])
lstm_prediction_result_reporting['order_date'] = pd.to_datetime(lstm_prediction_result_reporting['order_date'])

actual_result_reporting.to_csv('result/data/actual_results_reporting.csv', index = False)
lstm_prediction_result_reporting.to_csv('result/data/lstm_prediction_results_reporting.csv', index=False)

#### **Model Performance**

In [104]:
model_performance_df = pd.DataFrame(
                                    index =
                                      pd.MultiIndex.from_tuples(
                                          [
                                            ('LSTM', 'Train'),
                                            ('LSTM', 'Valid'),
                                            ('LSTM', 'Test')
                                          ]
                                      ),

                                    columns = ['MAE', 'MSE', 'R2-Score']
)

# LSTM Model Performance
model_performance_df.loc[('LSTM', 'Train'), 'MAE'] = lstm_mae_train
model_performance_df.loc[('LSTM', 'Valid'), 'MAE'] = lstm_mae_valid
model_performance_df.loc[('LSTM', 'Test'),  'MAE'] = lstm_mae_test

model_performance_df.loc[('LSTM', 'Train'), 'MSE'] = lstm_mse_train
model_performance_df.loc[('LSTM', 'Valid'), 'MSE'] = lstm_mse_valid
model_performance_df.loc[('LSTM', 'Test'), 'MSE']  = lstm_mse_test

model_performance_df.loc[('LSTM', 'Train'), 'R2-Score'] = lstm_r2s_train
model_performance_df.loc[('LSTM', 'Valid'), 'R2-Score'] = lstm_r2s_valid
model_performance_df.loc[('LSTM', 'Test'), 'R2-Score']  = lstm_r2s_test

model_performance_df.to_csv('result/data/lstm_performance.csv')

### **Data Visualization**

In [ ]:
plt.